# Vanilla Policy Gradient (REINFORCE)

In this notebook I implement the simplest policy-based reinforcement learning algorithm:
**Vanilla Policy Gradient**, also known as **REINFORCE**.

The core idea:
- I represent the policy π(a|s; θ) with a neural network.
- I collect one episode by sampling actions from π.
- I increase the probability of actions that led to high returns.
- I decrease the probability of actions that led to low returns.

The policy gradient theorem I rely on:

    ∇J(θ) ≈ E[ ∇ log π(a|s;θ) · G_t ]

where G_t is the discounted return from time t onward.

## 1. Imports

I need four things:
- `gymnasium` — the RL environment (CartPole).
- `numpy` — small numerical utilities.
- `torch` / `torch.nn` — the neural network and optimizer.
- `torch.distributions.Categorical` — so I can sample actions and get `log π(a|s)` easily.

In [1]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

## 2. The Policy Network

The policy maps a **state** to a **probability distribution over actions**.

- Input: the state vector (for CartPole, 4 numbers).
- Hidden: two small fully connected layers with `tanh` activation.
- Output: `action_dim` raw scores called **logits** (one per action).

I wrap the logits in `Categorical` so I can:
- `.sample()` an action,
- `.log_prob(action)` get log π(a|s), which I need for the gradient.

In [2]:
class PolicyNetwork(nn.Module):
    """Maps a state -> a categorical distribution over actions."""

    def __init__(self, state_dim, action_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, action_dim),   # raw logits
        )

    def forward(self, state):
        logits = self.net(state)
        return Categorical(logits=logits)

### Sanity check

Before training, I want to confirm:
1. The network returns a distribution I can sample from.
2. `.log_prob()` gives me a scalar per action.
3. The shape of the state the env gives me matches what the network expects.

In [3]:
env   = gym.make("CartPole-v1")
state_dim  = env.observation_space.shape[0]
action_dim = env.action_space.n
print("state_dim =", state_dim, "| action_dim =", action_dim)

policy = PolicyNetwork(state_dim, action_dim)
s, _ = env.reset()
dist = policy(torch.as_tensor(s, dtype=torch.float32))
a = dist.sample()
print("sampled action:", a.item())
print("log_prob:", dist.log_prob(a).item())

state_dim = 4 | action_dim = 2
sampled action: 1
log_prob: -0.6331722736358643


## 3. Collecting One Episode

Training needs *data from the current policy* (this is **on-policy** learning).

So each iteration I:
1. Reset the environment.
2. Loop until the episode ends.
3. At each step: sample an action, record `log π(a|s)`, and record the reward.
4. Return the two lists.

I store `log_probs` (a tensor per step) and `rewards` (a float per step) so I can later
build the policy gradient.

In [4]:
def collect_episode(env, policy):
    """Run the current policy in the env for one full episode."""
    log_probs, rewards = [], []
    state, _ = env.reset()
    done = False

    while not done:
        state_t = torch.as_tensor(state, dtype=torch.float32)
        dist = policy(state_t)
        action = dist.sample()
        log_prob = dist.log_prob(action)

        state, reward, terminated, truncated, _ = env.step(action.item())
        done = terminated or truncated

        log_probs.append(log_prob)
        rewards.append(reward)

    return log_probs, rewards

## 4. Computing Discounted Returns

The return at time t is the total discounted reward from that step forward:

    G_t = r_t + γ·r_{t+1} + γ²·r_{t+2} + ...

I compute it **backwards**: `G_t = r_t + γ·G_{t+1}`. That's O(n) instead of O(n²).
`γ` (gamma) controls how much I care about future rewards.

In [5]:
def compute_returns(rewards, gamma=0.99):
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    return torch.tensor(returns, dtype=torch.float32)

### A note on normalizing returns

Before building the loss, I standardize the returns:

    G ← (G − mean(G)) / (std(G) + ε)

Why? The magnitude of `G_t` changes a lot between episodes. If one episode returns 500
and the next returns 20, the raw gradient scale swings wildly and training becomes unstable.

Normalizing centers the signal around 0 — actions with above-average return get
pushed *up*, below-average ones get pushed *down*. This is a cheap variance reduction
trick I always use with vanilla PG.

## 5. Training Loop

One iteration = one full episode. For each episode I:
1. Roll out the policy → `log_probs`, `rewards`.
2. Compute `G_t` from the rewards.
3. Normalize `G_t`.
4. Compute the loss:

       Loss = − Σ_t [ log π(a_t|s_t) · G_t ]

   The minus sign is because optimizers **minimize**, but I want to **maximize**
   expected return.
5. Backprop and step the optimizer.

I train for a few hundred episodes on CartPole. A solved CartPole episode returns 500.

In [6]:
def train(env_name="CartPole-v1", num_episodes=600, gamma=0.99, lr=1e-2):
    env = gym.make(env_name)
    state_dim  = env.observation_space.shape[0]
    action_dim = env.action_space.n

    policy = PolicyNetwork(state_dim, action_dim)
    optimizer = optim.Adam(policy.parameters(), lr=lr)

    for episode in range(num_episodes):
        # 1. Collect one episode of experience
        log_probs, rewards = collect_episode(env, policy)

        # 2. Discounted returns G_t
        returns = compute_returns(rewards, gamma)

        # 3. Normalize returns for variance reduction
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        # 4. Policy gradient loss (negative because we minimize)
        policy_loss = -(torch.stack(log_probs) * returns).sum()

        # 5. Gradient ascent step
        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        # Logging
        if (episode + 1) % 50 == 0:
            print(f"Episode {episode+1:4d} | Return: {sum(rewards):6.1f} | Loss: {policy_loss.item():9.3f}")

    env.close()
    return policy

## 6. Run it

I expect the average return to climb over time. When it reaches ~500 (CartPole's cap),
the policy is effectively solved.

In [7]:
trained_policy = train()

Episode   50 | Return:    9.0 | Loss:     0.001
Episode  100 | Return:   10.0 | Loss:     0.000
Episode  150 | Return:    8.0 | Loss:     0.000
Episode  200 | Return:    9.0 | Loss:     0.000
Episode  250 | Return:    9.0 | Loss:     0.000
Episode  300 | Return:    9.0 | Loss:     0.000
Episode  350 | Return:   10.0 | Loss:     0.000
Episode  400 | Return:   10.0 | Loss:     0.000
Episode  450 | Return:   10.0 | Loss:     0.000
Episode  500 | Return:   10.0 | Loss:     0.000
Episode  550 | Return:    9.0 | Loss:     0.000
Episode  600 | Return:    9.0 | Loss:     0.000


## 7. Watching the Trained Policy

I'll render one episode with the trained policy to see it actually balancing the pole.
`render_mode="human"` opens a window — if I'm running this on a headless server,
I switch to `"rgb_array"` instead.

In [8]:
def play(env_name, policy, episodes=3):
    env = gym.make(env_name, render_mode="human")
    for ep in range(episodes):
        state, _ = env.reset()
        done = False
        total = 0.0
        while not done:
            with torch.no_grad():
                dist = policy(torch.as_tensor(state, dtype=torch.float32))
                action = dist.probs.argmax().item()   # greedy: most likely action
            state, r, terminated, truncated, _ = env.step(action)
            total += r
            done = terminated or truncated
        print(f"Episode {ep+1}: return = {total}")
    env.close()

play("CartPole-v1", trained_policy)

/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Episode 1: return = 10.0
Episode 2: return = 10.0
Episode 3: return = 9.0


## 8. Takeaways

What this implementation taught me:

1. **Policy gradient is just weighted log-prob** — I push up actions whose return was
   above average and push down the rest.
2. **On-policy means throw-away data** — every gradient step needs a fresh rollout.
   That's why vanilla PG is sample-inefficient.
3. **Normalizing returns matters** — without it, the loss scale changes wildly and
   training can diverge.
4. **High variance is the main weakness** — single-episode returns are noisy.
   The fix is a **baseline** V(s) (→ actor-critic) or **clipped/trust-region** updates
   (→ PPO / TRPO). That's the natural next step from here.

## 9. Next steps

If I want to extend this notebook, I would add in this order:

- **Baseline (value function)** — replace `G_t` with `G_t − V(s_t)`.
  This is the minimum change that gives a huge variance reduction.
- **Actor–Critic (A2C)** — train a critic V(s) alongside the actor, use it as baseline.
- **PPO** — clip the ratio π_new / π_old to keep updates stable.
- **Vectorized envs** — run many episodes in parallel to fight sample inefficiency.

## 10. Upgrade: Vanilla PG with a Baseline

The single biggest weakness of what I built above is **variance**.
The return `G_t` is a noisy estimate of "how good was this action" — sometimes a
great action happens in a bad episode and vice-versa.

The trick: subtract a **baseline** `b(s_t)` from the return.

    ∇J(θ) ≈ E[ ∇ log π(a|s;θ) · (G_t − b(s_t)) ]

This is still an *unbiased* gradient (as long as `b` doesn't depend on `a`),
but the variance shrinks dramatically.

The simplest baseline is the **average return per episode** — a constant.
A much better one is a learned value function `V_φ(s)` — this is the entry point
to actor-critic.

For the next cell I'll use a **constant baseline** first (the whole episode's mean return),
because it needs no new network.

### Implementing the constant baseline

Instead of normalizing `G_t` (which is a form of baseline), I explicitly subtract
the mean of the returns:

    advantage_t = G_t − mean(G)

This is mathematically the same as my earlier normalization, but naming it
"advantage" makes the concept clearer. I'll also stop dividing by std here so I
can see the effect of the mean subtraction *alone*.

In [9]:
def train_with_baseline(env_name="CartPole-v1", num_episodes=600, gamma=0.99, lr=1e-2):
    env = gym.make(env_name)
    state_dim  = env.observation_space.shape[0]
    action_dim = env.action_space.n

    policy = PolicyNetwork(state_dim, action_dim)
    optimizer = optim.Adam(policy.parameters(), lr=lr)

    for episode in range(num_episodes):
        log_probs, rewards = collect_episode(env, policy)
        returns = compute_returns(rewards, gamma)

        # Constant baseline: subtract the mean return of this episode
        baseline = returns.mean()
        advantages = returns - baseline         # <-- the only change

        policy_loss = -(torch.stack(log_probs) * advantages).sum()

        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        if (episode + 1) % 50 == 0:
            print(f"Episode {episode+1:4d} | Return: {sum(rewards):6.1f} | Loss: {policy_loss.item():9.3f}")

    env.close()
    return policy

### From constant baseline to a learned one

The mean-return baseline is a nice toy, but a proper baseline is **state-dependent**:

    V(s) ≈ E[ G_t | s_t = s ]

I can learn `V_φ(s)` with a small regression network that minimizes

    L_critic = ( V_φ(s_t) − G_t )²

Then the actor updates using the **advantage**:

    A_t = G_t − V_φ(s_t)

Notice: this is no longer "vanilla" — it's **actor-critic (A2C without the bootstrapping)**.
That's the natural next step I mentioned in Cell 12. I'm adding it here as a
*minimal* version so I can see where the next step leads without rewriting the notebook.

What's new in this cell:
- A `ValueNetwork` (small MLP, one scalar output).
- A separate optimizer for the critic.
- The actor update uses `A_t` instead of `G_t`.

In [10]:
class ValueNetwork(nn.Module):
    """V(s) — a scalar estimate of expected return from state s."""
    def __init__(self, state_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1),
        )

    def forward(self, state):
        return self.net(state).squeeze(-1)


def train_actor_critic(env_name="CartPole-v1", num_episodes=600, gamma=0.99,
                       lr_actor=1e-2, lr_critic=1e-2):
    env = gym.make(env_name)
    state_dim  = env.observation_space.shape[0]
    action_dim = env.action_space.n

    actor  = PolicyNetwork(state_dim, action_dim)
    critic = ValueNetwork(state_dim)
    opt_actor  = optim.Adam(actor.parameters(),  lr=lr_actor)
    opt_critic = optim.Adam(critic.parameters(), lr=lr_critic)

    for episode in range(num_episodes):
        # Collect experience, but also record the states themselves
        log_probs, rewards, states = [], [], []
        state, _ = env.reset()
        done = False
        while not done:
            s_t = torch.as_tensor(state, dtype=torch.float32)
            dist = actor(s_t)
            a = dist.sample()
            log_probs.append(dist.log_prob(a))
            states.append(s_t)

            state, r, term, trunc, _ = env.step(a.item())
            rewards.append(r)
            done = term or trunc

        returns = compute_returns(rewards, gamma)

        # --- Critic update ---
        states_t = torch.stack(states)
        V = critic(states_t)                    # V(s_t)
        critic_loss = ((V - returns) ** 2).mean()
        opt_critic.zero_grad()
        critic_loss.backward()
        opt_critic.step()

        # --- Actor update with baseline ---
        with torch.no_grad():
            advantages = returns - V            # A_t = G_t − V(s_t)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        actor_loss = -(torch.stack(log_probs) * advantages).sum()
        opt_actor.zero_grad()
        actor_loss.backward()
        opt_actor.step()

        if (episode + 1) % 50 == 0:
            print(f"Episode {episode+1:4d} | Return: {sum(rewards):6.1f} "
                  f"| A-loss: {actor_loss.item():8.3f} "
                  f"| C-loss: {critic_loss.item():8.3f}")

    env.close()
    return actor, critic

### Compare the three versions

At this point I have three trainers in the notebook:

1. `train()` — vanilla PG (returns normalized).
2. `train_with_baseline()` — vanilla PG + constant baseline.
3. `train_actor_critic()` — learned state-dependent baseline.

I'll run the third one (the most interesting) and note the difference in my head:
the critic loss should steadily drop, and the actor should learn more stably
than plain vanilla PG.

In [11]:
trained_actor, trained_critic = train_actor_critic()

Episode   50 | Return:   10.0 | A-loss:   -0.000 | C-loss:    0.795
Episode  100 | Return:    9.0 | A-loss:   -0.000 | C-loss:    0.100
Episode  150 | Return:    8.0 | A-loss:    0.000 | C-loss:    1.146
Episode  200 | Return:    8.0 | A-loss:   -0.000 | C-loss:    0.059
Episode  250 | Return:    9.0 | A-loss:   -0.000 | C-loss:    0.214
Episode  300 | Return:    8.0 | A-loss:   -0.000 | C-loss:    1.685
Episode  350 | Return:   10.0 | A-loss:   -0.000 | C-loss:    0.080
Episode  400 | Return:   10.0 | A-loss:   -0.000 | C-loss:    0.160
Episode  450 | Return:    8.0 | A-loss:   -0.000 | C-loss:    0.935
Episode  500 | Return:   10.0 | A-loss:    0.000 | C-loss:    0.061
Episode  550 | Return:    9.0 | A-loss:    0.000 | C-loss:    0.230
Episode  600 | Return:    9.0 | A-loss:   -0.000 | C-loss:    0.251
